In [ ]:
from pyspark.sql import SparkSession

spark_session = SparkSession.builder \
    .master("spark://192.168.2.156:7077") \
    .appName("RandomSampling") \
    .config("spark.dynamicAllocation.enabled", True) \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", True) \
    .config("spark.shuffle.service.enabled", False) \
    .config("spark.dynamicAllocation.executorIdleTimeout", "30s") \
    .config("spark.cores.max", 64) \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

file_path = "hdfs://192.168.2.156:9000/data/reddit/corpus-webis-tldr-17.json"
df = spark_session.read.json(file_path)

total_rows = df.count()
num_samples = 800000

if total_rows > num_samples:
    sample_fraction = num_samples / total_rows
    sampled_df = df.sample(False, sample_fraction, seed=42)
else:
    sampled_df = df


output_path = "hdfs://192.168.2.156:9000/output/Group31/reddit_sampled_800k.json"
sampled_df.coalesce(1).write.mode("overwrite").json(output_path)

spark_session.stop()

25/03/14 13:40:30 WARN StandaloneSchedulerBackend: Dynamic allocation enabled without spark.executor.cores explicitly set, you may get more executors allocated than expected. It's recommended to set spark.executor.cores explicitly. Please check SPARK-30299 for more details.
[Stage 0:=========================>                             (67 + 64) / 147]